# Week 3, day 1 (afternoon) — Worksheet 17: Bronze, silver, gold

Worksheet 16 joined the superstore tables. This sheet turns those joins into a
**pipeline**, using the same four files in `data/bronze/` and nothing else.

The **medallion architecture** names three layers, and the value of it is that
each one makes a promise the next is allowed to rely on:

| Layer | Contains | Promise |
|---|---|---|
| **Bronze** | source data as delivered, append-only, plus lineage | *nothing is lost* |
| **Silver** | cleaned, deduplicated, conformed, joined | *one row means one thing* |
| **Gold** | business aggregates, ready to consume | *a question has one answer* |

The temptation is always to skip silver — read bronze, group it, publish. Question
10 does exactly that and gets a wrong number.

Worksheet 16 found 40 duplicate `LineID` rows and dropped them without
explanation. This sheet finds out **where they came from**, which turns out to be
the reason bronze and silver are separate layers at all.

**Question 10 is supposed to raise an error.**

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 17 — Bronze, silver, gold. Run this once.
import glob
import pandas as pd

BRONZE = "data/bronze/"

print("the landing zone, as delivered:")
for path in sorted(glob.glob(BRONZE + "*.csv")):
    n = sum(1 for _ in open(path)) - 1
    print("  %-22s %5d rows" % (path.split("/")[-1], n))

BRONZE — take delivery, lose nothing

### Question 1

Build the bronze layer. Read every `orders_*.csv` and concatenate them, adding two lineage columns: `_source_file` (which file the row came from) and `_ingested_at` (a fixed timestamp). Print the shape and the row count per source file.
> **NOTE:** bronze does not clean. Do not deduplicate, do not fix types, do not join. Take delivery and record where each row came from.

In [ ]:
############################
## Your Code Here
############################

### Question 2

Bronze is append-only, so it can contain the same row twice. Find the duplicates: count rows whose `LineID` repeats, print which `_source_file` each copy came from, and the `OrderDate` range they cover.
> **NOTE:** before calling this corruption, look at *which files* the two copies arrived in.

In [ ]:
############################
## Your Code Here
############################

SILVER — one row means one thing

### Question 3

Deduplicate into the silver grain. Keep the **first** copy of each `LineID`, and print the row count before and after, plus `SUM(Sales)` before and after.
> **NOTE:** the difference in `SUM(Sales)` is what the re-delivery would have added to revenue.

In [ ]:
############################
## Your Code Here
############################

### Question 4

Conform it: join the customer and product dimensions onto silver with `how="left"` and `validate="many_to_one"`. Print the row count after each join and the columns gained.

In [ ]:
############################
## Your Code Here
############################

### Question 5

Add the return flag — **without** a join, using the lesson from worksheet 16 question 8. Print the row count (it must not change), the returned line count, and the returned order count.
> **NOTE:** `returns` is one row per order and silver is one row per line. A merge here fans out the returns side.

In [ ]:
############################
## Your Code Here
############################

### Question 6

Silver makes a promise, so test it. Write four assertions — `LineID` unique, no null join keys, no null dimension attributes after the joins, and row count equal to distinct bronze `LineID` — and print a PASS/FAIL line for each.
> **NOTE:** a layer whose promise is not checked is a layer whose promise is a hope.

In [ ]:
############################
## Your Code Here
############################

GOLD — one question, one answer

### Question 7

Build a gold table: revenue by `CustomerSegment` and `ProductCategory`, with the line count, total sales, and returned sales. Print it.

In [ ]:
############################
## Your Code Here
############################

### Question 8

Build a second gold table at a different grain — revenue by `Region` — then show why you must not join the two gold tables. Print the row count of a merge between them on nothing in common, versus the correct way to get both answers.
> **NOTE:** two aggregates at different grains have no key to join on. Going back to silver is the answer, not joining golds.

In [ ]:
############################
## Your Code Here
############################

### Question 9

Print the row count and `SUM(Sales)` at each layer — bronze, silver, and gold — as one table, so the pipeline reconciles end to end.
> **NOTE:** silver and gold must agree on the total. Bronze is allowed to differ, and you must be able to say by exactly how much and why.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Finally, skip silver: aggregate revenue by segment **straight from bronze**, and assert it equals the silver-based figure. **This is supposed to fail.** Read the difference.

In [ ]:
############################
## Your Code Here
############################